#### We are given a table called customer_state_log containing the following columns:
- cust_id
- state: The state of the session, where 1 indicates the session is active and 0 indicates the session has ended.
- timestamp: The timestamp when the state change occurred.
Our task is to calculate how many hours each user was active during the day based on the state transitions.

#### Solution 
- Window Specification: We define a window spec (window_spec) to partition the data by cust_id and order it by the timestamp. This helps in tracking the sessions for each customer in chronological order.
- LAG Function: Using the lag() function, we retrieve the timestamp of the previous row for each customer. This allows us to track when a session starts (state=1) and ends (state=0).
- Session Duration Calculation: We filter out rows where the state is 0 (indicating the session has ended), and calculate the session duration in minutes by subtracting the previous timestamp from the current timestamp.
- Group and Sum: The data is grouped by cust_id, and the total session duration (in minutes) is summed up for each customer.
- Convert to Hours: The total active minutes are converted into hours by dividing by 60.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
# Initialize Spark session
spark = SparkSession.builder.appName("CustomerSessionHours").getOrCreate()

In [0]:
# Sample data (as given in the problem)
data = [
    ('c001', 1, '07:00:00'),
    ('c001', 0, '09:30:00'),
    ('c001', 1, '12:00:00'),
    ('c001', 0, '14:30:00'),
    ('c002', 1, '08:00:00'),
    ('c002', 0, '09:30:00'),
    ('c002', 1, '11:00:00'),
    ('c002', 0, '12:30:00'),
    ('c002', 1, '15:00:00'),
    ('c002', 0, '16:30:00'),
    ('c003', 1, '09:00:00'),
    ('c003', 0, '10:30:00'),
    ('c004', 1, '10:00:00'),
    ('c004', 0, '10:30:00'),
    ('c004', 1, '14:00:00'),
    ('c004', 0, '15:30:00'),
    ('c005', 1, '10:00:00'),
    ('c005', 0, '14:30:00'),
    ('c005', 1, '15:30:00'),
    ('c005', 0, '18:30:00')
]

In [0]:
# Create a DataFrame
columns = ["cust_id", "state", "timestamp"]
df = spark.createDataFrame(data, columns)

In [0]:
# Convert 'timestamp' to a proper timestamp type (using 'HH:MM:SS' format)
df = df.withColumn("timestamp", F.col("timestamp").cast("timestamp"))

In [0]:
# Define a window specification to partition by cust_id and order by timestamp
window_spec = Window.partitionBy("cust_id").orderBy("timestamp")

In [0]:
# Calculate the previous timestamp for each row using LAG() function
df_with_lag = df.withColumn("prev_timestamp", F.lag("timestamp").over(window_spec))


In [0]:
# Filter out rows where state is 0 (session ends)
session_ends = df_with_lag.filter(df_with_lag.state == 0)


In [0]:
# Calculate session duration in minutes (difference between the session end time and start time)
session_duration = session_ends.withColumn(
    "session_duration_minutes",
    (F.unix_timestamp("timestamp") - F.unix_timestamp("prev_timestamp")) / 60
)

In [0]:
# Group by cust_id and calculate the total active time in hours
result = session_duration.groupBy("cust_id").agg(
    F.sum("session_duration_minutes").alias("total_active_minutes")
)

In [0]:
# Convert total active minutes to hours
result = result.withColumn("total_active_hours", F.col("total_active_minutes") / 60)


In [0]:
# Show the final result
result.show()

+-------+--------------------+------------------+
|cust_id|total_active_minutes|total_active_hours|
+-------+--------------------+------------------+
|   c001|               300.0|               5.0|
|   c002|               270.0|               4.5|
|   c003|                90.0|               1.5|
|   c004|               120.0|               2.0|
|   c005|               450.0|               7.5|
+-------+--------------------+------------------+

